# Train EMBER NGAV models with GPU 0

This notebook trains a partial EMBER 2018 subset for a 16GB RAM machine with a GTX1650 on GPU 0. It writes `models/ember_ngav.pkl` and `models/ember_anomaly.pkl`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

EMBER_PATH = PROJECT_ROOT / 'data' / 'ember_extracted' / 'ember2018'
OUTPUT_DIR = PROJECT_ROOT / 'models'

# Safe starting point for 16GB RAM and GTX1650 4GB VRAM.
LIMIT = 20_000
MAX_FILES = 1
SEED = 42
CONTAMINATION = 0.02
DEVICE = 'cuda'
GPU_ID = 0

EMBER_PATH, OUTPUT_DIR

## Check GPU and dataset

In [ ]:
import importlib.util
import subprocess

smi = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
print(smi.stdout if smi.returncode == 0 else smi.stderr)
print('xgboost installed:', importlib.util.find_spec('xgboost') is not None)
print('EMBER path exists:', EMBER_PATH.exists())
print('EMBER files:', sorted(p.name for p in EMBER_PATH.glob('*.jsonl')))

## Load a balanced partial subset

This reads only the first EMBER feature file by default and stops after the requested benign/malware sample count. Increase `LIMIT` or `MAX_FILES` only after the first run succeeds.

In [ ]:
from scripts.train_ember_models import load_subset

X, y = load_subset(
    EMBER_PATH,
    limit=LIMIT,
    seed=SEED,
    strategy='balanced',
    max_files=MAX_FILES,
)
print(f'samples={len(X)} benign={(y == 0).sum()} malware={(y == 1).sum()}')
print('example feature count:', len(X[0]) if X else 0)

## Train NGAV supervised model on GPU 0

In [ ]:
from scripts.train_ember_models import train_ember_ngav, save_bundle

ngav_pipeline, ngav_device = train_ember_ngav(X, y, device=DEVICE, gpu_id=GPU_ID)
ngav_threshold = 0.5 if ngav_device.startswith('cuda') else 0.0
save_bundle(
    OUTPUT_DIR / 'ember_ngav.pkl',
    ngav_pipeline,
    threshold=ngav_threshold,
    model_type='supervised',
    device=ngav_device,
)
print(f'saved {OUTPUT_DIR / "ember_ngav.pkl"} device={ngav_device} threshold={ngav_threshold}')

## Train benign-only anomaly detector

This model currently uses scikit-learn IsolationForest, so it runs on CPU while keeping the same partial subset in memory.

In [ ]:
from scripts.train_ember_models import train_ember_anomaly

anomaly_pipeline, anomaly_threshold = train_ember_anomaly(X, y, contamination=CONTAMINATION)
save_bundle(
    OUTPUT_DIR / 'ember_anomaly.pkl',
    anomaly_pipeline,
    threshold=anomaly_threshold,
    model_type='anomaly',
    device='cpu',
)
print(f'saved {OUTPUT_DIR / "ember_anomaly.pkl"} threshold={anomaly_threshold:.6f} device=cpu')

## Smoke test detector

In [ ]:
from server.detector import NgavDetector

detector = NgavDetector(include_behavior_model=False, include_ember_models=True)
sample_event = {
    'id': 'notebook-smoke-test',
    'endpoint': 'local-notebook',
    'event_type': 'pe_static',
    'data': {'ember_features': X[0]},
}
[d.to_dict() for d in detector.detect_event(sample_event)]